# Rename existing Elliot performance files

Chuyển đổi các file performance từ format timestamp cũ sang format chuẩn:

```
rec_cutoff_10_relthreshold_0_2026_06_20_01_24_11.tsv
  →  ItemKNN_cutoff_10.tsv
  →  VSM_cutoff_10.tsv

bestmodelparams_cutoff_10_relthreshold_0_2026_06_20.json
  →  bestmodelparams_ItemKNN_cutoff_10.json
```

Nếu có nhiều file cùng model-cutoff, file **mới nhất theo modified time** sẽ được giữ lại.

In [ ]:
# ── Cấu hình ──────────────────────────────────────────────────────────────────

RESULTS_ROOT = r"D:/recsys-pipeline/results/elliot"

# Đổi thành True để xóa file gốc sau khi rename thành công
DELETE_OLD_FILES = False

In [ ]:
import json
import re
from pathlib import Path

import pandas as pd


def normalize_model_name(raw: str) -> str:
    """Map full Elliot model string to short display name.

    'ItemKNN_nn=40_sim=cosine_...' -> 'ItemKNN'
    'VSM_sim=cosine_up=tfidf_...' -> 'VSM'
    'MostPop'                      -> 'MostPop'
    """
    base = raw.split("_")[0] if "_" in raw else raw
    return re.sub(r"[^a-zA-Z0-9]", "", base)


def process_performance_folder(perf_dir: Path, delete_old: bool) -> dict:
    """Process one performance/ folder. Returns stats dict."""
    stats = {"renamed": 0, "skipped": 0, "deleted": 0, "outputs": []}

    # ── TSV files: rec_cutoff_<N>_relthreshold_... ────────────────────────────
    # Sort ascending by mtime so newest overwrites older for same model-cutoff.
    tsv_files = sorted(
        perf_dir.glob("rec_cutoff_*.tsv"),
        key=lambda p: p.stat().st_mtime,
    )

    for tsv in tsv_files:
        m = re.search(r"rec_cutoff_(\d+)_", tsv.name)
        if not m:
            stats["skipped"] += 1
            continue
        cutoff = m.group(1)

        try:
            df = pd.read_csv(tsv, sep="\t")
        except Exception as e:
            print(f"  [WARN] Could not read {tsv.name}: {e}")
            stats["skipped"] += 1
            continue

        if "model" not in df.columns or df.empty:
            print(f"  [WARN] No 'model' column or empty: {tsv.name}")
            stats["skipped"] += 1
            continue

        ok = True
        for _, row in df.iterrows():
            model_name = normalize_model_name(str(row["model"]))
            out_path = perf_dir / f"{model_name}_cutoff_{cutoff}.tsv"
            try:
                pd.DataFrame([row]).to_csv(out_path, sep="\t", index=False)
                stats["renamed"] += 1
                stats["outputs"].append(str(out_path.relative_to(Path(RESULTS_ROOT).parent.parent)))
            except Exception as e:
                print(f"  [ERROR] Could not write {out_path.name}: {e}")
                ok = False

        if ok and delete_old:
            tsv.unlink()
            stats["deleted"] += 1

    # ── JSON files: bestmodelparams_cutoff_<N>_relthreshold_... ──────────────
    json_files = sorted(
        perf_dir.glob("bestmodelparams_cutoff_*.json"),
        key=lambda p: p.stat().st_mtime,
    )

    for jf in json_files:
        m = re.search(r"cutoff_(\d+)_", jf.name)
        if not m:
            stats["skipped"] += 1
            continue
        cutoff = m.group(1)

        try:
            with open(jf, encoding="utf-8") as f:
                data = json.load(f)
        except Exception as e:
            print(f"  [WARN] Could not read {jf.name}: {e}")
            stats["skipped"] += 1
            continue

        model_name = None
        for entry in data:
            if isinstance(entry, dict) and "recommender" in entry:
                model_name = normalize_model_name(entry["recommender"])
                break

        if model_name is None:
            print(f"  [WARN] No 'recommender' found in {jf.name}")
            stats["skipped"] += 1
            continue

        out_path = perf_dir / f"bestmodelparams_{model_name}_cutoff_{cutoff}.json"
        try:
            with open(out_path, "w", encoding="utf-8") as f:
                json.dump(data, f, indent=4)
            stats["renamed"] += 1
            stats["outputs"].append(str(out_path.relative_to(Path(RESULTS_ROOT).parent.parent)))
            if delete_old:
                jf.unlink()
                stats["deleted"] += 1
        except Exception as e:
            print(f"  [ERROR] Could not write {out_path.name}: {e}")

    return stats


print("Helper functions defined.")

In [ ]:
# ── Main processing loop ──────────────────────────────────────────────────────

results_root = Path(RESULTS_ROOT)
if not results_root.exists():
    raise FileNotFoundError(f"RESULTS_ROOT not found: {results_root}")

total_folders = 0
total_renamed = 0
total_skipped = 0
total_deleted = 0
all_outputs = []

for dataset_dir in sorted(results_root.iterdir()):
    if not dataset_dir.is_dir():
        continue
    perf_dir = dataset_dir / "performance"
    if not perf_dir.exists():
        continue

    # Check if there are any old-style files to process
    old_tsvs = list(perf_dir.glob("rec_cutoff_*.tsv"))
    old_jsons = list(perf_dir.glob("bestmodelparams_cutoff_*.json"))
    if not old_tsvs and not old_jsons:
        continue

    print(f"\n[{dataset_dir.name}]  ({len(old_tsvs)} TSV, {len(old_jsons)} JSON)")
    stats = process_performance_folder(perf_dir, delete_old=DELETE_OLD_FILES)

    total_folders += 1
    total_renamed += stats["renamed"]
    total_skipped += stats["skipped"]
    total_deleted += stats["deleted"]
    all_outputs.extend(stats["outputs"])

print("\nDone.")

In [ ]:
# ── Summary ───────────────────────────────────────────────────────────────────

print("=" * 60)
print(f"Folders processed : {total_folders}")
print(f"Files renamed     : {total_renamed}")
print(f"Files skipped     : {total_skipped}")
if DELETE_OLD_FILES:
    print(f"Old files deleted : {total_deleted}")
else:
    print(f"Old files deleted : 0  (DELETE_OLD_FILES=False, originals kept)")
print()
if all_outputs:
    print("Output files created/updated:")
    for p in sorted(set(all_outputs)):
        print(f"  {p}")
else:
    print("No output files created (nothing to rename).")